# Week 6 Exercise - The Price is Right (AdnanGobeljic)

This Week 6 contribution follows the course progression:
- load `ed-donner/items_lite`
- evaluate a constant baseline
- train a simple text + metadata regression model
- optionally compare with a zero-shot LLM pricer

Run from repo root or from `week6/`.

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import scipy.sparse as sp
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge


def resolve_week6_root() -> Path:
    current = Path.cwd().resolve()
    for base in [current, *current.parents]:
        candidate = base / "week6"
        if (candidate / "pricer" / "items.py").exists():
            return candidate
        if base.name == "week6" and (base / "pricer" / "items.py").exists():
            return base
    raise FileNotFoundError("Could not find week6/pricer from current working directory")


week6_root = resolve_week6_root()
sys.path.insert(0, str(week6_root))

from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print(f"Week 6 root: {week6_root}")
print("OpenAI key configured:" , bool(OPENAI_API_KEY))

In [ ]:
# Load curated course dataset from Hugging Face.

DATASET = "ed-donner/items_lite"
train, val, test = Item.from_hub(DATASET)

print(f"Train: {len(train):,} | Validation: {len(val):,} | Test: {len(test):,}")
print("Example item:", train[0])

In [ ]:
# Baseline model: always predict average training price.

MEAN_PRICE = float(np.mean([item.price for item in train]))
print(f"Mean training price: ${MEAN_PRICE:.2f}")


def constant_pricer(item: Item) -> float:
    return MEAN_PRICE

In [ ]:
# Evaluate baseline.

evaluate(constant_pricer, test, size=120)

In [ ]:
# Train a stronger local model (TF-IDF + metadata + Ridge).

TRAIN_LIMIT = 12000
train_subset = train[:TRAIN_LIMIT]


def item_to_text(item: Item) -> str:
    parts = [item.title or "", item.summary or "", item.category or ""]
    return "\n".join(parts)


def item_to_numeric(item: Item) -> list[float]:
    title = item.title or ""
    summary = item.summary or ""
    return [
        float(item.weight or 0.0),
        float(len(title)),
        float(len(summary)),
    ]


vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)
X_text = vectorizer.fit_transform([item_to_text(item) for item in train_subset])
X_num = sp.csr_matrix([item_to_numeric(item) for item in train_subset])
X_train = sp.hstack([X_text, X_num], format="csr")
y_train = np.array([item.price for item in train_subset], dtype=float)

ml_model = Ridge(alpha=1.0, random_state=42)
ml_model.fit(X_train, y_train)

print(f"Trained Ridge model on {len(train_subset):,} samples")

In [ ]:
def ml_pricer(item: Item) -> float:
    x_text = vectorizer.transform([item_to_text(item)])
    x_num = sp.csr_matrix([item_to_numeric(item)])
    x = sp.hstack([x_text, x_num], format="csr")
    prediction = float(ml_model.predict(x)[0])
    return float(np.clip(prediction, 0.0, 3000.0))


evaluate(ml_pricer, test, size=120)

In [24]:
# Optional zero-shot LLM predictor.

from openai import OpenAI

OPENAI_MODEL = "gpt-4.1-nano"
openai_client = OpenAI() if OPENAI_API_KEY else None

SYSTEM_PROMPT = """
You estimate e-commerce prices in USD.
Reply with only one number (no symbols or explanation).
"""


def llm_pricer(item: Item) -> float:
    if not openai_client:
        return ml_pricer(item)

    user_text = (item.summary or item.title or "")[:900]
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0,
        max_tokens=20,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Product details:\n{user_text}\n\nPredicted price:"},
        ],
    )

    text = (response.choices[0].message.content or "").strip()
    match = re.search(r"[-+]?\d*\.?\d+", text)
    if not match:
        return ml_pricer(item)

    return max(0.0, float(match.group()))

In [ ]:
# Keep LLM eval size small to control cost.

evaluate(llm_pricer, test, size=40)